In [8]:
# ============================================================
# CodeAlpha — Task 2: Exploratory Data Analysis (EDA)
# Dataset: Titanic
# ============================================================

# ---------- 1. Imports ----------
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')                       
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
import warnings

warnings.filterwarnings("ignore")           # silence palette/backend warnings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.max_open_warning'] = 0

# ---------- 2. Create folders ----------
IMG_DIR = "images"
DATA_DIR = "data"
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# ---------- 3. Load Dataset ----------
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())
print("\nData types & non-null counts:")
df.info()
print("\nNumerical summary:")
print(df.describe())
print("\nCategorical summary:")
print(df.describe(include='object'))

# ---------- 4. Data Quality ----------
print("\n" + "=" * 60)
print("DATA QUALITY")
print("=" * 60)

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("\nMissing values:")
print(pd.DataFrame({'Missing': missing,
                    'Percent': (missing / len(df) * 100).round(2)}))
print(f"\nDuplicate rows: {df.duplicated().sum()}")

num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"\nNumerical columns ({len(num_cols)}): {num_cols}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

# ---------- 5. Cleaning ----------
# Age -> median, Embarked -> mode, Cabin -> HasCabin flag
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['HasCabin'] = df['Cabin'].notna().astype(int)
df = df.drop(columns=['Cabin', 'Ticket', 'Name', 'PassengerId'])

# Refresh column lists after cleaning
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

# ---------- 6. Missing Values Heatmap ----------
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title("Missing Values Heatmap (after cleaning)")
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/01_missing_values.png", bbox_inches='tight')
plt.close()

# ---------- 7. Univariate Analysis — Numerical ----------
print("\n" + "=" * 60)
print("UNIVARIATE ANALYSIS")
print("=" * 60)

for col in num_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[col], kde=True, ax=axes[0], color='steelblue')
    axes[0].set_title(f"Distribution of {col}")
    sns.boxplot(x=df[col], ax=axes[1], color='salmon')
    axes[1].set_title(f"Boxplot of {col}")
    plt.tight_layout()
    plt.savefig(f"{IMG_DIR}/02_dist_{col}.png", bbox_inches='tight')
    plt.close()
    print(f"{col}: skew={df[col].skew():.2f}, kurtosis={df[col].kurt():.2f}")

# ---------- 8. Univariate Analysis — Categorical ----------
for col in cat_cols:
    if df[col].nunique() <= 20:
        plt.figure(figsize=(8, 4))
        df[col].value_counts().plot(kind='bar', color='teal')
        plt.title(f"Frequency of {col}")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig(f"{IMG_DIR}/03_count_{col}.png", bbox_inches='tight')
        plt.close()
        print(f"\n{col} value counts:")
        print(df[col].value_counts(normalize=True).round(3).head(10))

# ---------- 9. Bivariate — Correlation ----------
print("\n" + "=" * 60)
print("BIVARIATE ANALYSIS")
print("=" * 60)

corr = df[num_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/04_correlation_heatmap.png", bbox_inches='tight')
plt.close()

print("\nCorrelation with Survived:")
print(corr['Survived'].sort_values(ascending=False))

# ---------- 10. Survival Rate by Category (FIXED palette) ----------
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.barplot(data=df, x='Sex', y='Survived', ax=axes[0],
            hue='Sex', palette='Set2', legend=False)
axes[0].set_title("Survival Rate by Sex")

sns.barplot(data=df, x='Pclass', y='Survived', ax=axes[1],
            hue='Pclass', palette='Set2', legend=False)
axes[1].set_title("Survival Rate by Pclass")

sns.barplot(data=df, x='Embarked', y='Survived', ax=axes[2],
            hue='Embarked', palette='Set2', legend=False)
axes[2].set_title("Survival Rate by Embarked")

plt.tight_layout()
plt.savefig(f"{IMG_DIR}/05_survival_by_category.png", bbox_inches='tight')
plt.close()

# ---------- 11. Age & Fare distributions by Survival ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.kdeplot(data=df, x='Age', hue='Survived', fill=True, ax=axes[0])
axes[0].set_title("Age Distribution by Survival")
sns.kdeplot(data=df, x='Fare', hue='Survived', fill=True, ax=axes[1])
axes[1].set_title("Fare Distribution by Survival")
plt.tight_layout()
plt.savefig(f"{IMG_DIR}/06_age_fare_by_survival.png", bbox_inches='tight')
plt.close()

# ---------- 12. Hypothesis Testing ----------
print("\n" + "=" * 60)
print("HYPOTHESIS TESTS")
print("=" * 60)

# Chi-square: Sex vs Survived
ct = pd.crosstab(df['Sex'], df['Survived'])
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f"Chi-square (Sex vs Survived):    chi2={chi2:.2f}, p={p:.4e}")
print("  → Reject H0: Sex and survival are associated."
      if p < 0.05 else "  → Fail to reject H0.")

# Chi-square: Pclass vs Survived
ct2 = pd.crosstab(df['Pclass'], df['Survived'])
chi2, p, dof, _ = stats.chi2_contingency(ct2)
print(f"Chi-square (Pclass vs Survived): chi2={chi2:.2f}, p={p:.4e}")

# Chi-square: Embarked vs Survived
ct3 = pd.crosstab(df['Embarked'], df['Survived'])
chi2, p, dof, _ = stats.chi2_contingency(ct3)
print(f"Chi-square (Embarked vs Survived): chi2={chi2:.2f}, p={p:.4e}")

# T-test: Fare by Survival
a = df[df['Survived'] == 1]['Fare']
b = df[df['Survived'] == 0]['Fare']
t, p = stats.ttest_ind(a, b, equal_var=False)
print(f"T-test (Fare by Survival):       t={t:.3f}, p={p:.4e}")

# T-test: Age by Survival
a = df[df['Survived'] == 1]['Age']
b = df[df['Survived'] == 0]['Age']
t, p = stats.ttest_ind(a, b, equal_var=False)
print(f"T-test (Age by Survival):        t={t:.3f}, p={p:.4e}")

# ---------- 13. Outlier Detection (IQR) ----------
print("\n" + "=" * 60)
print("OUTLIERS (IQR method)")
print("=" * 60)

def detect_outliers_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return series[(series < lower) | (series > upper)].count(), lower, upper

for col in num_cols:
    n, lo, hi = detect_outliers_iqr(df[col])
    print(f"{col}: {n} outliers (bounds {lo:.2f} – {hi:.2f})")

# ---------- 14. Save Cleaned Data ----------
df.to_csv(f"{DATA_DIR}/titanic_cleaned.csv", index=False)
print(f"\n✅ Cleaned data saved to /{DATA_DIR}/titanic_cleaned.csv")

# ---------- 15. Key Insights ----------
print("\n" + "=" * 60)
print("KEY INSIGHTS")
print("=" * 60)
print("""
1. Dataset: 891 passengers × 12 features. Target: Survived (38.4% survived).

2. Missing data handled:
   - Cabin (77%)      -> dropped, replaced with HasCabin flag
   - Age (20%)        -> median imputation
   - Embarked (0.2%)  -> mode imputation

3. Strongest predictors of survival (by |correlation|):
   - Pclass    r = -0.34
   - HasCabin  r = +0.32
   - Fare      r = +0.26

4. Statistical significance confirmed:
   - Sex    x Survived: chi2 = 260.72, p < 0.001
   - Pclass x Survived: chi2 = 102.89, p < 0.001
   - Fare   x Survived: t    = 6.84,   p < 0.001
   - Age    x Survived: t    = -1.90,  p = 0.058 (not significant)

5. Fare is heavily right-skewed (skew = 4.79, kurtosis = 33.4) ->
   log transform recommended before modeling.

6. Recommendation: Sex, Pclass, Fare, and HasCabin should be the
   core features in any predictive model for this dataset.
""")

print(f"✅ All figures saved to /{IMG_DIR}/")

DATASET OVERVIEW
Shape: (891, 12)

First 5 rows:
   PassengerId  Survived  Pclass                                               Name     Sex   Age  SibSp  Parch            Ticket     Fare Cabin  \
0            1         0       3                            Braund, Mr. Owen Harris    male  22.0      1      0         A/5 21171   7.2500   NaN   
1            2         1       1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1      0          PC 17599  71.2833   C85   
2            3         1       3                             Heikkinen, Miss. Laina  female  26.0      0      0  STON/O2. 3101282   7.9250   NaN   
3            4         1       1       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1      0            113803  53.1000  C123   
4            5         0       3                           Allen, Mr. William Henry    male  35.0      0      0            373450   8.0500   NaN   

  Embarked  
0        S  
1        C  
2        S  
3        S